# Read the LEAD dataset

The dataset root comes from the repo's `.env`; point `DATA_ROOT` elsewhere to use another download. See [data access](../docs/data_access.md) for the full reference.

In [ ]:
from lead.common.env import read_dotenv

DATA_ROOT = read_dotenv("PY123D_DATA_ROOT", "data/lead/123D")

## Raw py123d access

In [ ]:
from py123d.api.scene.arrow.arrow_scene_builder import ArrowSceneBuilder
from py123d.api.scene.scene_filter import SceneFilter
from py123d.common.execution.thread_pool_executor import ThreadPoolExecutor

scenes = ArrowSceneBuilder(
    logs_root=f"{DATA_ROOT}/logs",
    maps_root=f"{DATA_ROOT}/maps",
).get_scenes(
    SceneFilter(
        split_names=["normal_view"],
        future_num_iterations=40,
        required_scene_modalities=["camera:all@initial"],
    ),
    ThreadPoolExecutor(),
)

print(f"{len(scenes)} scenes")
scene = scenes[0]

In [ ]:
from PIL import Image
from py123d.datatypes import LidarID

ego = scene.get_ego_state_se3_at_iteration(0)
boxes = scene.get_box_detections_se3_at_iteration(0)
lights = scene.get_traffic_light_detections_at_iteration(0)
lidar = scene.get_lidar_at_iteration(0, LidarID.LIDAR_TOP)
map_api = scene.get_map_api()

camera_ids = scene.get_camera_metadatas()
camera = scene.get_camera_at_iteration(0, next(iter(camera_ids)))
Image.fromarray(camera.rgb_image)

In [ ]:
meta = scene.get_custom_modality_at_iteration(0, "driving_meta").data
print(meta["scenario"], meta["target_speed"])
sorted(meta)[:10]

## LEAD loader

In [ ]:
from lead.log_reader import SceneLoader

loader = SceneLoader(
    DATA_ROOT,
    SceneFilter(future_num_iterations=40),
    perturbation_probability=0.0,
)

frame = loader[0]
print(f"{len(loader)} frames")
print(f"{len(frame.cameras)} cameras, target point {frame.target_point}")
Image.fromarray(frame.cameras[0].rgb_image)

In [ ]:
states = loader.read_future_ego_states(0, iterations=[5, 10, 15, 20, 25, 30, 35, 40])
states[40]